In [5]:
# ============================================================
# MODULE 9 — CHURN RISK SCORING
# ============================================================

import pandas as pd
import numpy as np



# ============================================================
# 1. LOAD CLEANED DATASETS
# ============================================================


customers = pd.read_csv(
    "cleaned_saas_customers.csv"
)

subscriptions = pd.read_csv(
    "cleaned_saas_subscriptions.csv"
)

usage = pd.read_csv(
    "cleaned_saas_usage.csv"
)

tickets = pd.read_csv(
    "cleaned_saas_tickets.csv"
)

print("=" * 70)
print("MODULE 9 — Churn Risk Scoring ")
print("=" * 70)

print("\nDatasets loaded successfully.")


print("Customers:", customers.shape)
print("Subscriptions:", subscriptions.shape)
print("Usage:", usage.shape)
print("Tickets:", tickets.shape)


MODULE 9 — Churn Risk Scoring 

Datasets loaded successfully.
Customers: (420, 8)
Subscriptions: (437, 9)
Usage: (4640, 8)
Tickets: (1379, 7)


In [7]:


# ============================================================
# 2. CONVERT NUMERIC COLUMNS
# ============================================================

subscriptions["MRR"] = pd.to_numeric(
    subscriptions["MRR"],
    errors="coerce"
)

subscriptions["Seats"] = pd.to_numeric(
    subscriptions["Seats"],
    errors="coerce"
)

usage["Logins"] = pd.to_numeric(
    usage["Logins"],
    errors="coerce"
)

usage["ActiveUsers"] = pd.to_numeric(
    usage["ActiveUsers"],
    errors="coerce"
)

usage["APICalls"] = pd.to_numeric(
    usage["APICalls"],
    errors="coerce"
)

usage["SessionMinutes"] = pd.to_numeric(
    usage["SessionMinutes"],
    errors="coerce"
)

tickets["ResolutionHours"] = pd.to_numeric(
    tickets["ResolutionHours"],
    errors="coerce"
)

tickets["SatisfactionScore"] = pd.to_numeric(
    tickets["SatisfactionScore"],
    errors="coerce"
)

print("Numeric conversion completed.")



Numeric conversion completed.


In [11]:

# ============================================================
# 3. CREATE CUSTOMER-LEVEL METRICS
# ============================================================

# -------------------------
# Subscription metrics
# -------------------------

subscription_metrics = (
    subscriptions
    .groupby("CustomerID")
    .agg(
        Total_MRR=("MRR", "sum"),
        Total_Seats=("Seats", "sum")
    )
    .reset_index()
)


# -------------------------
# Usage metrics
# -------------------------

usage_metrics = (
    usage
    .groupby("CustomerID")
    .agg(
        Total_Logins=("Logins", "sum"),
        Average_ActiveUsers=("ActiveUsers", "mean"),
        Total_SessionMinutes=("SessionMinutes", "sum"),
        Total_APICalls=("APICalls", "sum")
    )
    .reset_index()
)


# -------------------------
# Ticket metrics
# -------------------------

ticket_metrics = (
    tickets
    .groupby("CustomerID")
    .agg(
        Ticket_Count=("TicketID", "nunique"),
        Average_Satisfaction=("SatisfactionScore", "mean")
    )
    .reset_index()
)




In [10]:

# ============================================================
# 4. CREATE CUSTOMER RISK TABLE
# ============================================================

risk_data = customers[
    ["CustomerID", "CompanyName"]
].copy()

risk_data = risk_data.merge(
    subscription_metrics,
    on="CustomerID",
    how="left"
)

risk_data = risk_data.merge(
    usage_metrics,
    on="CustomerID",
    how="left"
)

risk_data = risk_data.merge(
    ticket_metrics,
    on="CustomerID",
    how="left"
)


# Customers without activity get zero activity

numeric_columns = [
    "Total_MRR",
    "Total_Seats",
    "Total_Logins",
    "Average_ActiveUsers",
    "Total_SessionMinutes",
    "Total_APICalls",
    "Ticket_Count",
    "Average_Satisfaction"
]

risk_data[numeric_columns] = (
    risk_data[numeric_columns]
    .fillna(0)
)


print("\nCustomer risk table created.")
print(risk_data.head())



Customer risk table created.
  CustomerID               CompanyName  Total_MRR  Total_Seats  Total_Logins  \
0      C1001      Ishaan Joshi Systems     278.60          6.0         148.0   
1      C1002  Tara Sharma Technologies     246.76          4.0         217.0   
2      C1003          Zoya Das Systems     278.60          6.0         187.0   
3      C1004           Diya Joshi Labs      80.36          9.0          82.0   
4      C1005        Tara Joshi Systems     618.76          4.0         133.0   

   Average_ActiveUsers  Total_SessionMinutes  Total_APICalls  Ticket_Count  \
0             4.500000                2241.4         10276.0           4.0   
1            12.181818                3021.1         15512.0           4.0   
2             7.250000                4734.2         19771.0           3.0   
3             4.200000                2530.0         14012.0           0.0   
4             4.692308                2102.0         13232.0           3.0   

   Average_Satisfact

In [12]:
# ============================================================
# 5. CREATE RISK SIGNALS
# ============================================================

# Median thresholds are used so that the risk rules
# are based on the behaviour of this dataset.

login_threshold = risk_data["Total_Logins"].median()

active_user_threshold = (
    risk_data["Average_ActiveUsers"].median()
)

session_threshold = (
    risk_data["Total_SessionMinutes"].median()
)

ticket_threshold = (
    risk_data["Ticket_Count"].median()
)

satisfaction_threshold = (
    risk_data["Average_Satisfaction"].median()
)


print("\nRisk thresholds:")
print("Login threshold:", login_threshold)
print("Active-user threshold:", active_user_threshold)
print("Session-minute threshold:", session_threshold)
print("Ticket threshold:", ticket_threshold)
print("Satisfaction threshold:", satisfaction_threshold)



Risk thresholds:
Login threshold: 82.0
Active-user threshold: 4.67948717948718
Session-minute threshold: 2165.8999999999996
Ticket threshold: 3.0
Satisfaction threshold: 5.5


In [13]:
# ============================================================
# 6. CALCULATE INDIVIDUAL RISK SIGNALS
# ============================================================

# Signal 1 — Low login activity

risk_data["Low_Login_Risk"] = np.where(
    risk_data["Total_Logins"] < login_threshold, 1,0)


# Signal 2 — Low active-user activity

risk_data["Low_ActiveUser_Risk"] = np.where(
    risk_data["Average_ActiveUsers"] < active_user_threshold,1,0)


# Signal 3 — Low session activity

risk_data["Low_Session_Risk"] = np.where(
    risk_data["Total_SessionMinutes"] < session_threshold,1, 0)


# Signal 4 — High ticket volume

risk_data["High_Ticket_Risk"] = np.where(
    risk_data["Ticket_Count"] > ticket_threshold,1,0)


# Signal 5 — Low satisfaction

# Only customers with recorded satisfaction
# are considered for this signal.

risk_data["Low_Satisfaction_Risk"] = np.where(
    (
        (risk_data["Average_Satisfaction"] > 0)
        &
        (
            risk_data["Average_Satisfaction"]
            < satisfaction_threshold
        )
    ),1,0)


In [14]:

# ============================================================
# 7. CREATE WEIGHTED RISK SCORE
# ============================================================

# Each signal contributes points.
#
# Low login       = 2 points
# Low active user = 1 point
# Low sessions    = 2 points
# High tickets    = 1 point
# Low satisfaction= 2 points
#
# Maximum = 8 points

risk_data["Risk_Score"] = (
    risk_data["Low_Login_Risk"] * 2
    +
    risk_data["Low_ActiveUser_Risk"] * 1
    +
    risk_data["Low_Session_Risk"] * 2
    +
    risk_data["High_Ticket_Risk"] * 1
    +
    risk_data["Low_Satisfaction_Risk"] * 2
)


print("\nRisk score distribution:")
print(
    risk_data["Risk_Score"]
    .value_counts()
    .sort_index()
)



Risk score distribution:
Risk_Score
0    38
1    59
2    33
3    64
4    41
5    70
6    49
7    47
8    19
Name: count, dtype: int64


In [15]:
# ============================================================
# 8. CLASSIFY CUSTOMERS BY RISK
# ============================================================

risk_data["Risk_Level"] = np.select(
    [
        risk_data["Risk_Score"] >= 6,
        risk_data["Risk_Score"] >= 3
    ],
    [
        "High Risk",
        "Medium Risk"
    ],
    default="Low Risk"
)


print("\nRisk level counts:")
print(
    risk_data["Risk_Level"]
    .value_counts()
)



Risk level counts:
Risk_Level
Medium Risk    175
Low Risk       130
High Risk      115
Name: count, dtype: int64


In [18]:
# ============================================================
# 9. RANK CUSTOMERS BY RISK
# ============================================================

risk_data = risk_data.sort_values(
    by=[
        "Risk_Score",
        "Total_MRR"
    ],
    ascending=[
        False,
        False
    ]
).reset_index(drop=True)


# Rank 1 = highest risk

risk_data["Risk_Rank"] = (
    risk_data.index + 1
)

# ============================================================
# 10. SHOW HIGHEST-RISK CUSTOMERS
# ============================================================

print("\nTOP 20 HIGHEST-RISK CUSTOMERS")

print(
    risk_data[
        [
            "Risk_Rank",
            "CustomerID",
            "CompanyName",
            "Total_MRR",
            "Risk_Score",
            "Risk_Level"
        ]
    ].head(20)
)




TOP 20 HIGHEST-RISK CUSTOMERS
    Risk_Rank CustomerID                CompanyName  Total_MRR  Risk_Score  \
0           1      C1385           Vihaan Das Group    2218.52           8   
1           2      C1257     Vihaan Patel Solutions    1858.76           8   
2           3      C1056          Kavya Rao Systems     658.68           8   
3           4      C1378      Sneha Reddy Solutions     383.20           8   
4           5      C1401     Naveen Gupta Solutions     342.28           8   
5           6      C1317        Divya Patel Systems     326.36           8   
6           7      C1398             Zoya Rao Group     326.36           8   
7           8      C1417  Naveen Patel Technologies     326.36           8   
8           9      C1074           Priya Khan Group     294.52           8   
9          10      C1152  Nikita Verma Technologies     294.52           8   
10         11      C1355  Nikita Reddy Technologies     294.52           8   
11         12      C1140         

In [21]:
# ============================================================
# 11. CALCULATE MRR IN HIGHEST-RISK GROUP
# ============================================================

high_risk_customers = risk_data[
    risk_data["Risk_Level"] == "High Risk"
].copy()


high_risk_mrr = (
    high_risk_customers["Total_MRR"]
    .sum()
)


total_mrr = (
    risk_data["Total_MRR"]
    .sum()
)


if total_mrr > 0:
    high_risk_mrr_percentage = (
        high_risk_mrr / total_mrr
    ) * 100
else:
    high_risk_mrr_percentage = 0


print("\n============================================================")
print("HIGHEST-RISK GROUP")
print("============================================================")

print(
    "High-risk customers:",
    len(high_risk_customers)
)

print(
    "MRR in high-risk group:",
    round(high_risk_mrr, 2)
)

print(
    "Percentage of total MRR:",
    round(high_risk_mrr_percentage, 2),
    "%"
)


HIGHEST-RISK GROUP
High-risk customers: 115
MRR in high-risk group: 71049.72
Percentage of total MRR: 26.98 %


In [22]:
# ============================================================
# 12. RISK SUMMARY
# ============================================================

risk_summary = (
    risk_data
    .groupby("Risk_Level")
    .agg(
        Customer_Count=("CustomerID", "count"),
        Total_MRR=("Total_MRR", "sum"),
        Average_MRR=("Total_MRR", "mean"),
        Average_Risk_Score=("Risk_Score", "mean")
    )
    .round(2)
)

# Put risk levels in logical order

risk_order = [
    "Low Risk",
    "Medium Risk",
    "High Risk"
]

risk_summary = (
    risk_summary
    .reindex(risk_order)
)

print("\nRISK SUMMARY")
print(risk_summary)


RISK SUMMARY
             Customer_Count  Total_MRR  Average_MRR  Average_Risk_Score
Risk_Level                                                             
Low Risk                130   83611.76       643.17                0.96
Medium Risk             175  108704.08       621.17                4.03
High Risk               115   71049.72       617.82                6.74


In [27]:
# 13. SAVE CUSTOMER RISK RANKING
# ============================================================

risk_data.to_csv(
    "module9_customer_churn_risk.csv",
    index=False
)

# ============================================================
# 14. SAVE RISK SUMMARY
# ============================================================

risk_summary.to_csv(
    "module9_risk_summary.csv"
)

print("\n============================================================")
print("MODULE 9 FILES SAVED")
print("============================================================")

print(
    "1. module9_customer_churn_risk.csv")

print(
    "2. module9_risk_summary.csv")




MODULE 9 FILES SAVED
1. module9_customer_churn_risk.csv
2. module9_risk_summary.csv


In [26]:
# ============================================================
# 15. FINAL MODULE 9 STATEMENT
# ============================================================

print("\nMODULE 9 CONCLUSION")

print(
    f"The highest-risk group contains "
    f"{len(high_risk_customers)} customers "
    f"with total MRR of "
    f"{high_risk_mrr:.2f}."
)

print(
    f"This represents "
    f"{high_risk_mrr_percentage:.2f}% "
    f"of total customer MRR."
)


MODULE 9 CONCLUSION
The highest-risk group contains 115 customers with total MRR of 71049.72.
This represents 26.98% of total customer MRR.
